#### 1. Set up the environment

In [19]:
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve().parent

USE_TORCH_241_CONFIG = True  # Needed for older machines
if USE_TORCH_241_CONFIG:
    install_script = "install_requirements_t241.py"
    requirements_file = "requirements_t241.yml"
else:
    install_script = "install_requirements.py"
    requirements_file = "requirements.yml"
INSTALL_SCRIPT = PROJECT_DIR / "setup" / install_script
REQUIREMENTS_FILE = PROJECT_DIR / "setup" / requirements_file

assert INSTALL_SCRIPT.exists(), INSTALL_SCRIPT
assert REQUIREMENTS_FILE.exists(), REQUIREMENTS_FILE

print("Project dir:", PROJECT_DIR)
print("Install script:", INSTALL_SCRIPT)
print("Requirements:", REQUIREMENTS_FILE)

!python "{INSTALL_SCRIPT}" --requirements "{REQUIREMENTS_FILE}" --project-dir "{PROJECT_DIR}"

Project dir: C:\Users\salat\Alessandro Salatiello - Alljoined
Install script: C:\Users\salat\Alessandro Salatiello - Alljoined\setup\install_requirements_t241.py
Requirements: C:\Users\salat\Alessandro Salatiello - Alljoined\setup\requirements_t241.yml

Project directory: C:\Users\salat\Alessandro Salatiello - Alljoined
Git repos directory: C:\Users\salat\Alessandro Salatiello - Alljoined\code

Python environment:
  executable: c:\Users\salat\anaconda3\envs\reve-allj\python.exe
  version:    3.11.15

No git section found. Skipping repo clone.

Initial torch status:
  version:        2.4.1+cu118
  cuda build:     True
  cuda version:   11.8
  cuda available: True
  gpu:            NVIDIA T500
  import path:    c:\Users\salat\anaconda3\envs\reve-allj\Lib\site-packages\torch\__init__.py

Torch already satisfies requirements. Skipping torch reinstall.

Installing regular packages:
  - numpy
  - scipy
  - matplotlib
  - pandas
  - scikit-learn
  - pyyaml
  - packaging
  - tqdm
  - ipywidget

#### 2. Preprocess data and train models

In [ ]:
# Imports modules
import importlib
import torch
import os
import utils

importlib.reload(utils)
from huggingface_hub import login

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONHASHSEED"] = "0"
read_data, prep, dls, mods, trainers = utils.import_modules(
    ["read_data", "preprocessing", "dataloaders", "models", "trainers"]
)

# Load HF Token
login(token=utils.get_env("HF_TOKEN"))

# =============== Main configuration variables =======================================================
# Enable TRAINING mode to train the models (if disabled, the trained model checkpoints will be used)
TRAINING = True

# Enable hyperparameter optimization (if disabled, the default values in train_optuna.yml will be used)
HP_OPT = True

# Enable DEBUG mode to run the pipeline on a small data subset
DEBUG = False

# Subtrial mode
SUBTRIAL = True

# Selected models
SELECTED_MODELS = ["cbramod"]  # Any of 'cbramod','reve','unishape'
# ====================================================================================================

# Set device, seed, and batch_size
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
seed = 42
batch_size = 2
subtrial_size = 1  # Number of seconds in one subtrial

# Read data, metadata, and configuration files
(
    data,
    labels,
    label2id,
    id2label,
    trial_info,
    channels,
    sf,
    configs,
    paths,
) = read_data.read(SUBTRIAL=SUBTRIAL)

# Main preprocessing and training loop
models = list(configs.keys())
assert all([sm in models for sm in SELECTED_MODELS]), (
    f"Please selectect one of more models among {models}"
)
models = list(set(models) & set(SELECTED_MODELS))
print(f"Selected models: {models}")

metrics = {}
ypred_test = {}
for model in models:
    # Set random seed to ensure reproducibility
    print(f"\nHandling model '{model}' {'*' * 60}")
    utils.set_seeds(seed)

    # Run model-specific preprocessing pipeline
    print(f"Preprocessing {'*' * 68}")
    (
        data,
        data_spec,
        labels,
        channels,
        trials,
        n_subtrials_per_trial,
    ) = prep.preprocess(
        data=data,
        model_config=configs[model]["preprocessing"],
        sf=sf,
        DEBUG=DEBUG,
        SUBTRIAL=SUBTRIAL,
        labels=labels,
        channels=channels,
        subtrial_size=subtrial_size,
    )

    # Create dataloaders
    print(f"Creating trainin/validation/test dataloaders {'*' * 38}")
    dataloaders = dls.make_dataloaders(data, labels, batch_size, seed)

    # Train models (after hp optimization)
    if TRAINING:
        print(f"Training models{'*' * 68}")
        # print(f"Running hyperparameter optimization{'*' * 38}")
        (
            best_model,
            metrics[model],
            best_hyperparams,
        ) = trainers.optimize_model(
            model=model,
            dataloaders=dataloaders,
            channels=channels,
            data_spec=data_spec,
            n_classes=len(label2id),
            HP_OPT=HP_OPT,
            device=device,
            home_folder=paths["results"],
        )

    else:
        print(f"Retrieving best trained model{'*' * 38}")
        (
            best_model,
            metrics[model],
            best_hyperparams,
        ) = trainers.retrieve_best_model(
            model=model,
            channels=channels,
            data_spec=data_spec,
            n_classes=len(label2id),
            device=device,
            home_fld=paths["results"] / "models",
        )

    # Evaluate best model on test set
    print(f"Evaluating best model on test set{'*' * 38}")
    ypred_test[model] = trainers.evaluate_model(
        model=best_model,
        dataloader=dataloaders["test"],
        device=device,
        dummy_y=True,
    )["ypred"]

    # Convert subtrial predictions to trial predictions
    if SUBTRIAL:
        ypred_test[model] = prep.subtrials2trials(
            ypred_test[model],
            trials["test"],
            n_subtrials_per_trial["test"],
            modality="mode",
        )

    # Save best model predictions
    utils.save_preds(
        model,
        ypred_test[model],
        id2label,
        results_dir=paths["results"],
    )

    print(metrics[model])
    print(best_hyperparams)
    print()

# Print training summary and create summary plot
summary_df, plot_path = utils.summarize_model_metrics(
    metrics=metrics,
    dataloaders=dataloaders,
    n_classes=len(label2id),
)

C:\Users\salat\Alessandro Salatiello - Alljoined
Imported latest version of 'read_data' module
Imported latest version of 'preprocessing' module
Imported latest version of 'dataloaders' module
Imported latest version of 'models' module
Imported latest version of 'trainers' module


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Selected models: ['cbramod']

Handling model 'cbramod' ************************************************************
Preprocessing ********************************************************************
Filtering...


KeyboardInterrupt: 